# My RAG pipeline

**What RAG is, in one line:** you turn your documents into numbers (embeddings), store them, then when a question comes in you turn the question into numbers too, find the closest document chunks, and hand those chunks to an LLM as context so it answers from *your* data instead of guessing.

Four stages, each one a function:

1. **Load** files from `docs/`
2. **Split** them into chunks and **embed** each chunk into a vector store (Chroma)
3. **Retrieve** the chunks nearest to a query
4. **Generate** an answer with an LLM using those chunks

Each stage is defined once and used by everything below it. The only cells that "do" anything are the ones at the bottom.

In [16]:
import os, glob, hashlib
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings # LLM
from langchain_chroma import Chroma # Vector Store
from langchain.chat_models import init_chat_model

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "put GOOGLE_API_KEY=... in .env"

# all parameters in one place
DOC_DIR = "docs"
DB_DIR = "./chroma_db"
CHUNK_SIZE, CHUNK_OVERLAP = 800, 120
EMBED_MODEL = "nomic-embed-text"
LLM_MODEL, LLM_PROVIDER = "llama3.2", "ollama" # Substitute for another LLM

## 1. Load

A LangChain **Document** is `page_content` (the text) plus `metadata` (a dict, usually the source path). Loaders turn files into Documents. `.md`, `.txt` and `.pdf` are handled; add more loaders as needed.

`source` metadata is what lets you tell the user which file an answer came from.

In [17]:
LOADERS = {
    ".pdf": lambda p: PyPDFLoader(p),                    # one Document per page
    ".md":  lambda p: TextLoader(p, encoding="utf-8"),
    ".txt": lambda p: TextLoader(p, encoding="utf-8"),
}

def load_docs(folder=DOC_DIR):
    docs = []
    for path in sorted(glob.glob(f"{folder}/**/*", recursive=True)):
        ext = os.path.splitext(path)[1].lower()
        if ext in LOADERS:
            docs += LOADERS[ext](path).load()
    assert docs, f"no readable files under {os.path.abspath(folder)}"
    return docs

## 2. Split and embed

**Why split:** embedding models have a token limit, and a whole file as one vector is too blurry to match a specific question. Chunks of a few hundred characters each get their own vector.

`RecursiveCharacterTextSplitter` cuts on paragraphs first, then lines, sentences, words, so chunks stay coherent. `chunk_overlap` repeats the tail of one chunk at the head of the next so a sentence split across the boundary still lands whole in at least one chunk.

**Embedding** maps text to a fixed-length vector (768 numbers for `nomic-embed-text`). Semantically similar text ends up geometrically close.

**Chroma** stores those vectors on disk and does the nearest-neighbour search.

**Why deterministic ids:** each chunk gets an id hashed from `source + chunk index`. `add_documents` with ids is an upsert, so re-running this after adding or editing a file updates the store instead of duplicating it or, worse, skipping it. The earlier version used "skip if the store is non-empty", which is why your second file never got indexed: the store was built once with old data and the guard kept it frozen.

Changing `CHUNK_SIZE` or `EMBED_MODEL` changes what a chunk is, so pass `rebuild=True` in that case to wipe first.

In [18]:
def chunk_id(doc, i):
    return hashlib.md5(f"{doc.metadata['source']}::{i}".encode()).hexdigest()

def split(docs, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=overlap)
    return splitter.split_documents(docs)   # metadata carried onto every chunk

def get_store():
    return Chroma(
        collection_name="docs",
        embedding_function=OllamaEmbeddings(model=EMBED_MODEL),
        persist_directory=DB_DIR,
    )

def index(rebuild=False):
    store = get_store()
    if rebuild:
        store.delete_collection()
        store = get_store()
    chunks = split(load_docs())
    ids = [chunk_id(d, i) for i, d in enumerate(chunks)]
    store.add_documents(chunks, ids=ids)     # upsert, safe to re-run
    print(f"{len(chunks)} chunks from {len({d.metadata['source'] for d in chunks})} files, "
          f"store now holds {store._collection.count()}")
    return store

## 3. Retrieve

The query goes through the **same** embedding model as the chunks. Different models produce incompatible vector spaces. Chroma returns the `k` chunks with the smallest distance.

The score is a **distance**, so lower means more similar. `k` is *how many* chunks come back, not how similar they must be; use `similarity_search_with_relevance_scores` plus a threshold if you want a cutoff.

In [19]:
def retrieve(store, question, k=4): # K value determines how similar chunks should be
    return store.similarity_search_with_score(question, k=k)

def show(hits):
    for doc, score in hits:
        print(f"[{score:.3f}] {doc.metadata['source']}")
        print("   ", doc.page_content[:200].replace(chr(10), " "), "\n")

## 4. Generate

Stuff the retrieved chunks into the prompt and tell the model to answer from them only. That instruction is what stops it falling back on training data when your docs don't cover the question.

`init_chat_model` is provider-agnostic. `("llama3.2", model_provider="ollama")` switches to a fully local generator.

Note `.text` rather than `.content`: newer Gemini responses return `content` as a list of blocks (text plus a signature), `.text` flattens that to a string.

In [20]:
llm = init_chat_model(LLM_MODEL, model_provider=LLM_PROVIDER)

def build_prompt(question, hits):
    ctx = "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d, _ in hits)
    return (f"Answer using only the context below. Cite the [source] you used. "
            f"If the context does not contain the answer, say so.\n\n"
            f"Context:\n{ctx}\n\nQuestion: {question}")

def ask(store, question, k=4, debug=False):
    hits = retrieve(store, question, k)
    if debug:
        show(hits)
    return llm.invoke(build_prompt(question, hits)).text # API req context + prompt

In [21]:
store = index()            # index(rebuild=True) after changing chunk size or embed model

23 chunks from 2 files, store now holds 23


In [23]:
print(ask(store, "which method takes the longest", debug=True))

[0.967] docs/data.md
    # Coffee brewing  ## Espresso Espresso uses about 9 bars of pressure and a fine grind. A double shot pulls in 25 to 30 seconds from 18g of coffee, yielding roughly 36g in the cup. Grind too coarse and 

[1.133] docs/me.md
    ---  ## 9. Open items requiring an answer  2. **Sigma compensation** — equity, monthly payment, or both. 3. **Novelty Bakery** — was he paid, and did he invoice. 4. **Live Stripe transactions** — conf 

[1.142] docs/data.md
    ## Storage Whole beans keep for about a month after roast in an airtight container away from light. Ground coffee goes stale within days. Never refrigerate, the moisture ruins it. """ 

[1.142] docs/me.md
    **Payment: UNRESOLVED.** Original facts said he was paid to build it. Later he said he offered it on the house and the owner offered payment. Whether money changed hands is unconfirmed.  ### Machine L 

According to the context, the Pour over method takes the longest, with a total brew time of around three minu

In [24]:
print(ask(store, "who am i"))

According to the context, you are Sheikh R Ahmed.


## Where to go next

- **Eval set first.** Ten questions with expected source file. Measure how often the right chunk lands in the top k before tuning anything.
- **Chunk size tuning.** 800/120 is a starting point. Smaller chunks give precise matches but less context per hit; larger the reverse. `index(rebuild=True)` after each change.
- **Hybrid search.** Vector search misses exact keywords (names, numbers). Add BM25 (`langchain_community.retrievers.BM25Retriever`) and merge results.
- **Reranking.** Retrieve 20, rerank to 4 with a cross-encoder.
- **Metadata filters.** `store.similarity_search(q, filter={"source": "docs/me.md"})` restricts the search.
- **Swap the embedder to HuggingFace** once your environment is stable: `langchain_huggingface.HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")`. Rebuild the store when you do.